# Llama Metrics

## Joint TKG Quintuples and Triples Calculation

In [ ]:
from metrics import sample_quintuple_compare, sample_triple_compare
from sklearn.metrics import f1_score
from post_processing import get_data, truth_quintuples_and_triples_preprocess, pred_quintuple_preprocess

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_preprocess)
tkg_llama_preds = get_data("llama3-8B-tkg-preds.json", pred_quintuple_preprocess)

In [ ]:
strict_quin_results = []
relaxed_quin_results = []
triple_results = []
for truth, pred in zip(test_data, tkg_llama_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), list(pred['quintuples'].values()))
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    strict_quin_results.extend(strict_out)
    relaxed_quin_results.extend(relaxed_out)
    triple_results.extend(trips_out)

{"Quin relaxed":f1_score([1]*len(relaxed_quin_results), relaxed_quin_results), "Quin strict":f1_score([1]*len(strict_quin_results), strict_quin_results), "Trips Acc":f1_score([1]*len(triple_results), triple_results)}

## Isolated ET NER

In [ ]:
from metrics import sample_ner_compare, get_ner_scores
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")
et_ner_llama_preds = get_data("llama3-8B-et-ner-preds.json")

In [ ]:
strict_ner_et_results = []
relaxed_ner_et_results = []

for pred, truth in zip(et_ner_llama_preds, test_data):
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['pred'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Geo NER

In [ ]:
from post_processing import geo_eval_formating, get_data
from metrics import sample_ner_compare, get_ner_scores

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json", preprocessor=geo_eval_formating)
geo_ner_preds = get_data("D:\\GeoTKG\\predictions\\llama3-8B-geoner-preds.json")

In [ ]:
strict_geo_results = []
relaxed_geo_results = []
for pred, truth in zip(geo_ner_preds, test_data):
    strict, relaxed = sample_ner_compare(truth, pred['pred'], geo_ner=True)
    strict_geo_results.extend(strict)
    relaxed_geo_results.extend(relaxed)
get_ner_scores(strict_geo_results, relaxed_geo_results)

# GeoTKG Metrics

In [ ]:
from Pipeline import GeoTKGPipeline
import json
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

out_preds = []
process_sep = []
model = GeoTKGPipeline()
batch_size = 2
for i in range(0, len(test_data), batch_size):
    samples = test_data[i:i+batch_size]
    for j, sample in enumerate(samples):
        if len(sample['text'])>50:
            process_sep.append(test_data.index(sample))
            samples.pop(j)
    dcts = [inst['value'] for sample in samples for inst in sample['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in sample['text'] for wrd in sent]) for sample in samples]
    output = model.pred(text, dcts, return_ner_results=True)
    out_preds.extend(output)
    print(f"Processed {len(out_preds)}/{len(test_data)}")

In [ ]:
process_sep
for pred_i in process_sep:
    dcts = [inst['value'] for inst in test_data[pred_i]['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in test_data[pred_i]['text'] for wrd in sent])]
    output = model.pred(text, dcts, return_ner_results=True)
    out_preds.insert(pred_i, output[0])

In [ ]:
from copy import deepcopy
backup = deepcopy(out_preds)

## Joint TKG Quintuples and Triples Calculation

In [ ]:
from metrics import sample_quintuple_compare, sample_triple_compare
from sklearn.metrics import f1_score
from post_processing import get_data, truth_quintuples_and_triples_formating

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_formating)

strict_results = []
relaxed_results = []
triples_results = []
for truth, pred in zip(test_data, out_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), pred['quintuples'])
    relaxed_results.extend(relaxed_out)
    strict_results.extend(strict_out)
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    triples_results.extend(trips_out)

{"Quin relaxed":f1_score([1]*len(relaxed_results), relaxed_results), "Quin strict":f1_score([1]*len(strict_results), strict_results), "triples":f1_score([1]*len(triples_results), triples_results)}

## Isolated Geo NER

In [4]:
from geotkg.models.GeoEntityModel import GeoEntityModel
import torch
from post_processing import get_data, geo_eval_formating
from transformers import AutoTokenizer
from metrics import sample_ner_compare, get_ner_scores

TOKENIZER = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

GeoNER = GeoEntityModel(base="roberta-base").to(device="cuda")
load = torch.load("geotkg\\results\\geo_model\\geo_model.pt")
GeoNER.load_state_dict(load['model_state_dict'])

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json")

strict_ner_et_results = []
relaxed_ner_et_results = []
for sample in test_data:
    text = " ".join(sample["tokens"]).strip()
    (times, entities), tokens = GeoNER.predict(text, return_tokens=True)
    ids = tokens['input_ids'][0]
    instances = [[TOKENIZER.decode(ids[inst[0]:inst[1]], skip_special_tokens=True).strip(), inst[2][2:]] for inst in [*times[0],*entities[0]]]
    truth_instances = geo_eval_formating(sample)
    strict_results, relaxed_results = sample_ner_compare(truth_instances, instances, geo_ner=True)
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)



d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initializ

{'strict_text': 0.6793498803074209,
 'relaxed_text': 0.7731741573033708,
 'type': 0.7413544668587896}

## Isolated ET NER

In [ ]:
from post_processing import get_data
from metrics import sample_ner_compare, get_ner_scores

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

In [ ]:
strict_ner_et_results = []
relaxed_ner_et_results = []
for pred, truth in zip(out_preds, test_data):
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['events']+pred['times'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Normalisation
Done in training file

## Isolated ET Linking and EE Temporal Relations
Done in training file